# Sampling and Cleaning Data

***
**This file will sample small data from each data source, to define a cleaning pipeline for each data source to be used when aggregating all the data.**
***
Terminology you will see throughout this data:
- LSOA - Lower Layer Super Output Area - Is a government mandated area of land for a small area, picked with the aim to keep a relatively stable population size within.
- LAD - Local Authority District - Is an area of land used for subnational administration. Each LAD will contain one or many LSOA's.
- PFA - Police Force Area - Is an area of land that contains multiple LAD's which a single police force will dictate.
*** 
There are 5 main sources of data for this project:  
- Location Lookup Table -> to relate location codes of different grain
- Population Data -> to denote the population of each LSOA
- Deprivation Data -> to denote the deprivation of each LSOA
- Crime Severity Data -> to denote the severity of each crime such that more dangerous areas can be spotted
- Crime Data -> Each individual crime committed per police region, with area of the LSOA attached
  
***

## Sampling Data

This file will go through the processes of taking a sample from the raw data, analysing it, and defining a cleaning process.

In [1]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

***
***
### Location Lookup Table

#### Ingestion

This lookup table is composed of two tables:
1) LSOA -> LAD conversion table         link: __https://ckan.publishing.service.gov.uk/dataset/local-authority-district-to-community-safety-partnership-to-pfa-april-2025-lookup-in-ew/resource/e8cc60d8-f3bb-4a29-ad58-821247d88d95__

2) LAD -> PFA conversion table          link: __https://ckan.publishing.service.gov.uk/dataset/lsoa-2021-to-electoral-ward-2024-to-lad-2024-best-fit-lookup-in-ew__

This section will sample and define a cleaning process for the data, such that is can be merged and aggregated in later files.

In [2]:
# Import LSOA <-> LAD data:
raw_lsoa_lad = pd.read_csv('../Data/Raw/lookup/lsoa-lad.csv')

raw_lsoa_lad.head(5)

,LSOA21CD,LSOA21NM,LSOA21NMW,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,ObjectId
0,E01012000,Hartlepool 007E,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,1
1,E01011964,Hartlepool 007B,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,2
2,E01011999,Hartlepool 007D,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,3
3,E01011967,Hartlepool 007C,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,4
4,E01011951,Hartlepool 007A,NaN,E05013038,Burn Valley,NaN,E06000001,Hartlepool,NaN,5


In [3]:
# Import LAD <-> PFA raw data
raw_lad_pfa = pd.read_csv('../Data/Raw/lookup/lad-pfa.csv')

raw_lad_pfa.head(5)

,LAD25CD,LAD25NM,CSP25CD,CSP25NM,PFA25CD,PFA25NM,ObjectId
0,E06000058,"Bournemouth, Christchurch and Poole",E22000367,Dorset,E23000039,Dorset,1
1,E06000059,Dorset,E22000367,Dorset,E23000039,Dorset,2
2,E06000060,Buckinghamshire,E22000303,Aylesbury Vale,E23000029,Thames Valley,3
3,E06000060,Buckinghamshire,E22000306,Chiltern,E23000029,Thames Valley,4
4,E06000060,Buckinghamshire,E22000311,South Bucks,E23000029,Thames Valley,5


***
#### Cleaning and Validation

Firstly, note the year of intake for each table:  
LSOA -> LAD: 2021 LSOA, **2024 LAD**  
LAD -> PFA: **2025 LAD**, 2025 PFA  
This is important as the LAD areas of Barnsley and Sheffield were updated in 2025.  
**Any data before 2025 will have the old regions, and old codes for these areas.**  
***
Secondly, note the useless columns for the lookup table that is to be produced.  
**In order to reduce the workload, dropping unnecessary data will reduce the workload.**

In [4]:
# Drop useless columns, and rename the columns left

lsoa_lad = raw_lsoa_lad[['LSOA21CD', 'LSOA21NM', 'LAD24CD', 'LAD24NM']]

lsoa_lad = lsoa_lad.rename(columns={
    'LSOA21CD': 'lsoa_code',
    'LSOA21NM': 'lsoa_name',
    'LAD24CD': 'lad_code',
    'LAD24NM': 'lad_name'
})

lsoa_lad.head()

,lsoa_code,lsoa_name,lad_code,lad_name
0,E01012000,Hartlepool 007E,E06000001,Hartlepool
1,E01011964,Hartlepool 007B,E06000001,Hartlepool
2,E01011999,Hartlepool 007D,E06000001,Hartlepool
3,E01011967,Hartlepool 007C,E06000001,Hartlepool
4,E01011951,Hartlepool 007A,E06000001,Hartlepool


In [5]:
# Drop useless columns, and rename the columns left

lad_pfa = raw_lad_pfa[['LAD25CD', 'LAD25NM', 'PFA25CD', 'PFA25NM']]

lad_pfa = lad_pfa.rename(columns={
    'LAD25CD': 'lad_code',
    'LAD25NM': 'lad_name',
    'PFA25CD': 'pfa_code',
    'PFA25NM': 'pfa_name'
})

lad_pfa.head()

,lad_code,lad_name,pfa_code,pfa_name
0,E06000058,"Bournemouth, Christchurch and Poole",E23000039,Dorset
1,E06000059,Dorset,E23000039,Dorset
2,E06000060,Buckinghamshire,E23000029,Thames Valley
3,E06000060,Buckinghamshire,E23000029,Thames Valley
4,E06000060,Buckinghamshire,E23000029,Thames Valley


*Check overall structure and size*

In [6]:
# Check data types and overall size
lsoa_lad.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35672 entries, 0 to 35671
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lsoa_code  35672 non-null  object
 1   lsoa_name  35672 non-null  object
 2   lad_code   35672 non-null  object
 3   lad_name   35672 non-null  object
dtypes: object(4)
memory usage: 1.1+ MB


4 columns  
35672 rows  
Correct data types.

In [7]:
# Check data types and overall size
lad_pfa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332 entries, 0 to 331
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   lad_code  332 non-null    object
 1   lad_name  332 non-null    object
 2   pfa_code  332 non-null    object
 3   pfa_name  332 non-null    object
dtypes: object(4)
memory usage: 10.5+ KB


4 columns  
332 rows  
Correct data types.

In [8]:
# Check for null values
lsoa_lad.isnull().sum()

lsoa_code    0
lsoa_name    0
lad_code     0
lad_name     0
dtype: int64

No nulls.

In [9]:
# Check for null values
lad_pfa.isnull().sum()

lad_code    0
lad_name    0
pfa_code    0
pfa_name    0
dtype: int64

No nulls.

In [10]:
# Check for duplicated data
print(lsoa_lad.duplicated().sum())

0


No duplicate data.

In [11]:
# Check for duplicated data
print(lad_pfa.duplicated().sum())

14


14 pieces of duplicate data.  
Look at them to see if there's a reason for the error:

In [12]:
display(lad_pfa[lad_pfa.duplicated()])

,lad_code,lad_name,pfa_code,pfa_name
3,E06000060,Buckinghamshire,E23000029,Thames Valley
4,E06000060,Buckinghamshire,E23000029,Thames Valley
5,E06000060,Buckinghamshire,E23000029,Thames Valley
7,E06000061,North Northamptonshire,E23000022,Northamptonshire
8,E06000061,North Northamptonshire,E23000022,Northamptonshire
9,E06000061,North Northamptonshire,E23000022,Northamptonshire
11,E06000062,West Northamptonshire,E23000022,Northamptonshire
13,E06000063,Cumberland,E23000002,Cumbria
14,E06000063,Cumberland,E23000002,Cumbria
16,E06000064,Westmorland and Furness,E23000002,Cumbria


These entries are all entirely duplicated within the dataframe.  
It doesn't seem as if there is a major reasoning behind this duplicity, it is likely due to the in between column (for voting regions) splitting LAD's in different ways to PFA's.
There is no need to keep them, they provide no extra context to the project.  
**Conclusion:** drop duplicates, note the number dropped and the reason.

In [13]:
dropped_rows = {}

print(f'The number of rows before dropping duplicates: {lad_pfa.shape[0]}')

dropped_rows['duplicates'] = lad_pfa.duplicated().sum()

lad_pfa = lad_pfa.drop_duplicates()

print(f'The number of rows after dropping duplicates: {lad_pfa.shape[0]}')

print('Reasons and counts for dropped rows:')
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

The number of rows before dropping duplicates: 332
The number of rows after dropping duplicates: 318
Reasons and counts for dropped rows:
duplicates: 14


***
These Databases are now ready to merge.  
Function to clean will be as follows:

In [14]:
def Clean_LsoaLad(raw_lsoa_lad, dropped_rows):
    lsoa_lad = raw_lsoa_lad[['LSOA21CD', 'LSOA21NM', 'LAD24CD', 'LAD24NM']]

    lsoa_lad = lsoa_lad.rename(columns={
        'LSOA21CD': 'lsoa_code',
        'LSOA21NM': 'lsoa_name',
        'LAD24CD': 'lad_code',
        'LAD24NM': 'lad_name'
    })

    return lsoa_lad

def Clean_LadPfa(raw_lad_pfa, dropped_rows):
    lad_pfa = raw_lad_pfa[['LAD25CD', 'LAD25NM', 'PFA25CD', 'PFA25NM']]

    lad_pfa = lad_pfa.rename(columns={
        'LAD25CD': 'lad_code',
        'LAD25NM': 'lad_name',
        'PFA25CD': 'pfa_code',
        'PFA25NM': 'pfa_name'
    })

    dropped_rows['duplicates'] = lad_pfa.duplicated().sum()

    lad_pfa = lad_pfa.drop_duplicates()

    return lad_pfa, dropped_rows

To check how many rows have been dropped after a function:

In [15]:
def Print_DroppedRows(dropped_rows):
    print('\nReason and count of dropped rows:')

    tot_dropped = 0
    for reason in dropped_rows:
        tot_dropped += dropped_rows[reason]
        print(f'{reason}: {dropped_rows[reason]}')

    print(f'total rows dropped: {tot_dropped}')

***
***
### Population Table

#### Ingestion

This table is made from ONS estimates for population within LSOA's of england, from the years 2022-2024. This means it will need some extrapolation in order to have 2025 and 2026 data.

link: __https://www.ons.gov.uk/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/lowersuperoutputareamidyearpopulationestimates__

<div class="alert alert-block alert-warning">
<b>Warning:</b> This section may take some time to import, there are 35,000 rows per year from the excel spreadsheet. It usually takes ~1min 30seconds to complete the import.
</div>

This file will sample only 1 year's worth of data - 2022.

In [16]:
# Import Population Data Raw
raw_pop_2022 = pd.read_excel(f'../Data/Raw/population/population.xlsx', sheet_name='Mid-2022 LSOA 2021', skiprows=3, usecols=['LAD 2023 Code', 'LAD 2023 Name', 'LSOA 2021 Code', 'LSOA 2021 Name', 'Total'])

raw_pop_2022.head()

,LAD 2023 Code,LAD 2023 Name,LSOA 2021 Code,LSOA 2021 Name,Total
0,E06000001,Hartlepool,E01011949,Hartlepool 009A,1876
1,E06000001,Hartlepool,E01011950,Hartlepool 008A,1117
2,E06000001,Hartlepool,E01011951,Hartlepool 007A,1260
3,E06000001,Hartlepool,E01011952,Hartlepool 002A,1635
4,E06000001,Hartlepool,E01011953,Hartlepool 002B,1984


#### Cleaning and Validation

In [17]:
# Remove useless columns
# As lookup table will handle all location details, only keep the lsoa code

pop_2022 = raw_pop_2022[['LSOA 2021 Code', 'Total']]

pop_2022.head()

,LSOA 2021 Code,Total
0,E01011949,1876
1,E01011950,1117
2,E01011951,1260
3,E01011952,1635
4,E01011953,1984


In [18]:
pop_2022.info

<bound method DataFrame.info of       LSOA 2021 Code  Total
0          E01011949   1876
1          E01011950   1117
2          E01011951   1260
3          E01011952   1635
4          E01011953   1984
...              ...    ...
35667      W01001324   1888
35668      W01001898   1452
35669      W01001959   1547
35670      W01001960   1481
35671      W01001961   2321

[35672 rows x 2 columns]>

Correct datatypes.

In [19]:
pop_2022.isnull().sum()

LSOA 2021 Code    0
Total             0
dtype: int64

No null values.

In [20]:
print(pop_2022.duplicated().sum())

0


No duplicate values.

***
Data is fully clean.  
The function to clean it is as follows:

In [21]:
def Clean_Population(raw_population, year):
    # Drop useless columns
    population = raw_population[['LSOA 2021 Code', 'Total']]

    # Rename columns
    population = population.rename(columns={
        'LSOA 2021 Code': 'lsoa_code',
        'Total': 'population',
    })

    return population    

This assumes all population data to use 2021 LSOA codes (which they luckily do)

***
***
### Deprivation Table

#### Ingestion

This data is sourced from ONS and relates to how deprived certain areas are. It holds information on deprivation of different kinds, and an overall deprivation score.

link: __https://www.gov.uk/csv-preview/691ded56d140bbbaa59a2a7d/File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv__

In [22]:
# Import Deprivation Data Raw
raw_depr = pd.read_csv(f'../Data/Raw/deprivation/deprivation.csv')

raw_depr.head()

,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.400,9136,3,1837,397,225,1260


***
#### Cleaning and Validation

In [23]:
# Remove unnecessary columns and rename those left

depr = raw_depr[[
    'LSOA code (2021)', 
    'Index of Multiple Deprivation (IMD) Score',
    'Income Score (rate)',
    'Employment Score (rate)',
    'Education, Skills and Training Score',
    'Barriers to Housing and Services Score'
]]

depr = depr.rename(columns={
    'LSOA code (2021)': 'lsoa_code',
    'Index of Multiple Deprivation (IMD) Score': 'imd_score',
    'Income Score (rate)': 'incm_score',
    'Employment Score (rate)': 'empl_score',
    'Education, Skills and Training Score': 'edcn_score',
    'Barriers to Housing and Services Score': 'hous_score'
})

depr.head()

,lsoa_code,imd_score,incm_score,empl_score,edcn_score,hous_score
0,E01000001,8.742,0.013,0.014,0.004,10.950
1,E01000002,4.722,0.018,0.010,0.169,6.703
2,E01000003,9.250,0.107,0.064,3.269,9.735
3,E01000005,19.884,0.211,0.104,17.852,24.623
4,E01000006,25.307,0.343,0.120,25.442,38.025


In [24]:
depr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33755 entries, 0 to 33754
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   lsoa_code   33755 non-null  object 
 1   imd_score   33755 non-null  float64
 2   incm_score  33755 non-null  float64
 3   empl_score  33755 non-null  float64
 4   edcn_score  33755 non-null  float64
 5   hous_score  33755 non-null  float64
dtypes: float64(5), object(1)
memory usage: 1.5+ MB


Correct data types.

In [25]:
# Check for null values
depr.isnull().sum()

lsoa_code     0
imd_score     0
incm_score    0
empl_score    0
edcn_score    0
hous_score    0
dtype: int64

No null values.

In [26]:
# Check for duplicated data
depr.duplicated().sum()

np.int64(0)

No duplicated values.

***
Data is now clean.  
Function to clean it will look as follows:

In [27]:
def Clean_Deprivation(raw_depr):
    # Get useful columns
    depr = raw_depr[[
        'LSOA code (2021)', 
        'Index of Multiple Deprivation (IMD) Score',
        'Income Score (rate)',
        'Employment Score (rate)',
        'Education, Skills and Training Score',
        'Barriers to Housing and Services Score'
    ]]

    # Rename columns
    depr = depr.rename(columns={
        'LSOA code (2021)': 'lsoa_code',
        'Index of Multiple Deprivation (IMD) Score': 'imd_score',
        'Income Score (rate)': 'incm_score',
        'Employment Score (rate)': 'empl_score',
        'Education, Skills and Training Score': 'edcn_score',
        'Barriers to Housing and Services Score': 'hous_score'
    })

    return depr


***
***
### Crime Severity

#### Ingestion

This dataset uses data from the ONS for crime weighting and data from data.police.uk for the crime categories.  
Unfortunately, the names do not perfectly line up. This meant that initial sorting of the data had to be done manually.  
The only data added was the 'Crime Category' column which relates the two datasets used.  
The raw data is held in the raw data files, but the manually sorted data is stored in the processed files.  

link for ONS crime severity weighting: __https://www.ons.gov.uk/peoplepopulationandcommunity/crimeandjustice/datasets/crimeseverityscoredatatool__  
link for data.police.uk category data: __https://data.police.uk/static/files/police-uk-category-mappings.csv__

In [28]:
# Import crime severity categorised data set
raw_sev = pd.read_csv(f'../Data/Processed/crime-severity-raw/crime-severity-categorised.csv')

raw_sev.head()

,Crime Index,Offence,Weight,Crime Category
0,"1, 4.1/10/2",Homicide,"7,979",Violence and sexual offences
1,2,Attempted murder,"4,663",Violence and sexual offences
2,4.3,Intentional destruction of viable unborn child,15,Violence and sexual offences
3,4.4,Causing death or serious injury by dangerous d...,"1,092",Violence and sexual offences
4,4.6,Causing death by careless driving when under t...,"1,595",Violence and sexual offences


#### Cleaning and Validation

In [29]:
# Check data types and overall size
raw_sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Crime Index     245 non-null    object
 1   Offence         250 non-null    object
 2   Weight          250 non-null    object
 3   Crime Category  242 non-null    object
dtypes: object(4)
memory usage: 7.9+ KB


Weight should be held as a float, not an object.

In [30]:
# Check for nulls
raw_sev.isnull().sum()

Crime Index       5
Offence           0
Weight            0
Crime Category    8
dtype: int64

**Crime Category has null values.**
- Set crime category nulls to 'No Category', such that they are known as null.   
- Looking at the data, these offenses are related to cyber risks and hacking. These will not be found within the crime dataset, as it only logs in person activities.  
**Conclusion**: Set null values to 'No Category'
  
**Crime Index has null values**
- This column will be dropped eventually, but it represents the different crime types within offences, it is required as to not observe duplicates when none are present.
**Conclusion:** Set null values to 'No Index'

In [31]:
# Check for duplicates
print(raw_sev.duplicated().sum())

0


No duplicate data.

In [32]:
# Change weight to integer
raw_sev['Weight'] = raw_sev['Weight'].astype(str).str.replace(',', '', regex=False)
raw_sev['Weight'] = pd.to_numeric(raw_sev['Weight'], errors='coerce')

print(f'Weight datatype: {raw_sev['Weight'].dtype}')

Weight datatype: int64


In [33]:
# Set null values of Crime Category to 'No Category'
raw_sev['Crime Category'] = raw_sev['Crime Category'].fillna('No Category')

print(raw_sev['Crime Category'].value_counts())

Crime Category
Other crime                     93
Violence and sexual offences    83
Burglary                        16
Criminal damage and arson       12
Public order                     9
Other theft                      8
No Category                      8
Possession of weapons            7
Drugs                            5
Vehicle crime                    4
Robbery                          2
Theft from the person            1
Bicycle Theft                    1
Shoplifting                      1
Name: count, dtype: int64


In [34]:
# Set null values of Crime Index to 'No Index'
raw_sev['Crime Index'] = raw_sev['Crime Index'].fillna('No Index')

print(raw_sev['Crime Index'].value_counts())

Crime Index
No Index       5
21             2
1, 4.1/10/2    1
86             1
67             1
              ..
28H            1
29             1
29A            1
30, 30A        1
NFIB 90        1
Name: count, Length: 245, dtype: int64


In [35]:
raw_sev.isnull().sum()

Crime Index       0
Offence           0
Weight            0
Crime Category    0
dtype: int64

No null values.

In [36]:
raw_sev.duplicated().sum()

np.int64(0)

No duplicates.

In [37]:
# Rename columns

raw_sev = raw_sev.rename(columns={
    'Crime Category': 'crime_cat',
    'Weight': 'weight'
})

***
Data is now clean to be transformed.  
Data cleaning process is as follows:

In [38]:
def Clean_Severity(raw_severity):
    # Change weight to integer
    raw_severity['Weight'] = raw_severity['Weight'].astype(str).str.replace(',', '', regex=False)
    raw_severity['Weight'] = pd.to_numeric(raw_severity['Weight'], errors='coerce')

    # Set null values of Crime Category to 'No Category'
    raw_sev['Crime Category'] = raw_sev['Crime Category'].fillna('No Category')

    # Set null values of Crime Index to 'No Index'
    raw_sev['Crime Index'] = raw_sev['Crime Index'].fillna('No Index')

    # Rename columns
    raw_sev = raw_sev.rename(columns={
        'Crime Category': 'crime_cat',
        'Weight': 'weight'
    })

    return raw_sev

***
***
### Crime Data

#### Ingestion

This data is taken from data.police.uk and denotes every crime within the uk caught in person. Crimes such as hacking and fraud, that occur online are not often reported in this data. This project will use data from 4 large cities in northern England - merseyside (Liverpool), west midlands (birmingham), west yorkshire (leeds), and south yorkshire (sheffield).  
The data is split monthly, so this file will use a small sample of 1 month's data to define a cleaning process.

link: __https://data.police.uk/data/__

In [39]:
## Import one months worth of data
police_region = 'south-yorkshire'
year_month = '2026-03' # Get the most recent data available

sth_yk = pd.read_csv(f'../Data/Raw/crime-data/{police_region}/{year_month}-{police_region}-street.csv')

sth_yk.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.397019,53.054017,On or near Supermarket,E01019456,Amber Valley 005E,Anti-social behaviour,NaN,NaN
1,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.451708,53.599208,On or near Jack Close Orchard,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
2,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.454483,53.598502,On or near B6428,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
3,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.446274,53.603435,On or near Warren Close,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN
4,NaN,2026-03,South Yorkshire Police,South Yorkshire Police,-1.450879,53.602548,On or near Ruston Drive,E01007434,Barnsley 001A,Anti-social behaviour,NaN,NaN


#### Cleaning and Validation

In [40]:
# Looking at data
sth_yk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15313 entries, 0 to 15312
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               12867 non-null  object 
 1   Month                  15313 non-null  object 
 2   Reported by            15313 non-null  object 
 3   Falls within           15313 non-null  object 
 4   Longitude              14645 non-null  float64
 5   Latitude               14645 non-null  float64
 6   Location               15313 non-null  object 
 7   LSOA code              14645 non-null  object 
 8   LSOA name              14645 non-null  object 
 9   Crime type             15313 non-null  object 
 10  Last outcome category  12867 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 1.4+ MB


***
Data types are **not** all correct.  
Month should be a date.
Context should be an object. (also it will be dropped so doesn't matter.)

**Conclusion:** Set month to be held as a date.
***

In [41]:
sth_yk.isnull().sum()

Crime ID                  2446
Month                        0
Reported by                  0
Falls within                 0
Longitude                  668
Latitude                   668
Location                     0
LSOA code                  668
LSOA name                  668
Crime type                   0
Last outcome category     2446
Context                  15313
dtype: int64

***
**Crime ID has null values** - This means we need to create a new primary key for this database.  
As each database is categorised by its location and its year and month, we will use that in its primary key.  
eg: mers_2026_03_00001 - This allows for up to 100,000 crimes per month per police region.  
  
Last outcome category has null values. This will eventually be dropped, so it is not a worry.  
context is fully null, it will eventually be dropped, so it is not a worry.
  
**Conclusion:** Create a new Crime ID column, drop last outcome and context.
***
**LSOA Code and Name, and lat/long have null values** - Take a look at where these come from:

In [42]:
display(sth_yk[sth_yk['LSOA code'].isnull()])

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
14645,385c2d968a19f5171229e2b004267b002d4eb5890e9a26...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Bicycle theft,Investigation complete; no suspect identified,NaN
14646,0482b49cd0ed02d50e081c3299513a251c7a7eaa402d53...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Bicycle theft,Investigation complete; no suspect identified,NaN
14647,ee877e2ce8520054c50ff4b8133ad507b262517619be9a...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Burglary,Unable to prosecute suspect,NaN
14648,1e4d119293955d7fcf5deb1c54217e180fc80405bfb792...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Burglary,Investigation complete; no suspect identified,NaN
14649,4469629a0b5276698861f49afcd0bcd30532cb13dc6063...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Burglary,Under investigation,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
15308,5b62c6378634daf74552dd19fd42c7b805f3674752e975...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Investigation complete; no suspect identified,NaN
15309,ccd37130ebbf0b7eb866028b8c74ab6932097d5ad3cf1d...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Action to be taken by another organisation,NaN
15310,14e4a6a0562adf62ba3895665e72c403f619211b1ff92e...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Unable to prosecute suspect,NaN
15311,ec63fbdabb11a93010334a77d79dc67c5b39f77ceb6e7b...,2026-03,South Yorkshire Police,South Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Other crime,Unable to prosecute suspect,NaN


seemingly no reason for the missing location, as no ideas where these may have gone - they will be dropped.  
**Conclusion:** drop rows with null location values.

In [43]:
# Check for duplicates
sth_yk.duplicated().sum()

np.int64(658)

Duplicate values can be dropped fully, they have just been entered into the database multiple times.
***

In [44]:
# Drop useless columns

sth_yk = sth_yk[['Crime ID', 'LSOA code', 'LSOA name', 'Month', 'Latitude', 'Longitude', 'Crime type']]

sth_yk.head()

,Crime ID,LSOA code,LSOA name,Month,Latitude,Longitude,Crime type
0,NaN,E01019456,Amber Valley 005E,2026-03,53.054017,-1.397019,Anti-social behaviour
1,NaN,E01007434,Barnsley 001A,2026-03,53.599208,-1.451708,Anti-social behaviour
2,NaN,E01007434,Barnsley 001A,2026-03,53.598502,-1.454483,Anti-social behaviour
3,NaN,E01007434,Barnsley 001A,2026-03,53.603435,-1.446274,Anti-social behaviour
4,NaN,E01007434,Barnsley 001A,2026-03,53.602548,-1.450879,Anti-social behaviour


In [45]:
# Rename columns
sth_yk = sth_yk.rename(columns={
    'Crime ID': 'old_crime_id',
    'Month': 'date',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'LSOA code': 'lsoa_code',
    'LSOA name': 'lsoa_name',
    'Crime type': 'crime_cat'
})

sth_yk.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat
0,NaN,E01019456,Amber Valley 005E,2026-03,53.054017,-1.397019,Anti-social behaviour
1,NaN,E01007434,Barnsley 001A,2026-03,53.599208,-1.451708,Anti-social behaviour
2,NaN,E01007434,Barnsley 001A,2026-03,53.598502,-1.454483,Anti-social behaviour
3,NaN,E01007434,Barnsley 001A,2026-03,53.603435,-1.446274,Anti-social behaviour
4,NaN,E01007434,Barnsley 001A,2026-03,53.602548,-1.450879,Anti-social behaviour


In [46]:
# Change month to date
sth_yk['date'] = pd.to_datetime(sth_yk['date'], format='%Y-%m')

sth_yk.head()

,old_crime_id,lsoa_code,lsoa_name,date,latitude,longitude,crime_cat
0,NaN,E01019456,Amber Valley 005E,2026-03-01,53.054017,-1.397019,Anti-social behaviour
1,NaN,E01007434,Barnsley 001A,2026-03-01,53.599208,-1.451708,Anti-social behaviour
2,NaN,E01007434,Barnsley 001A,2026-03-01,53.598502,-1.454483,Anti-social behaviour
3,NaN,E01007434,Barnsley 001A,2026-03-01,53.603435,-1.446274,Anti-social behaviour
4,NaN,E01007434,Barnsley 001A,2026-03-01,53.602548,-1.450879,Anti-social behaviour


In [47]:
# Drop duplicate rows
dropped_rows = {}

print(f'row count before dropping: {sth_yk.shape[0]}')

dropped_rows['duplicates'] = sth_yk.duplicated().sum()

sth_yk = sth_yk.drop_duplicates()
print(f'number of duplicates after dropping: {sth_yk.duplicated().sum()}')

print(f'row count after dropping: {sth_yk.shape[0]}')

row count before dropping: 15313
number of duplicates after dropping: 0
row count after dropping: 14655


In [48]:
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

duplicates: 658


In [49]:
# Fill old crime ID with 'No ID' for crimes with no ID
sth_yk['old_crime_id'] = sth_yk['old_crime_id'].fillna('No ID')

print(sth_yk['old_crime_id'].value_counts())

old_crime_id
No ID                                                               1788
7d45b84fc109da923c8da4502f25d85d09f72809e2cf21f2117fc01bf4c9c349       1
d22f65d6b58e1b60afb8fdea350d848063d39c2f342bf563028a3d7acf9b8c3d       1
de15dbc5b4ae0590119bf3d84abc946531bc605ab086e10b95f74a8291ae8982       1
6168b8c293960a2139f6d3ab489e6a025b59429d210aa8a5ae408dfe67605fb8       1
                                                                    ... 
a17f7b3238f772cca0fe8853f631b3ddaed45483dde406c609b970453c1f8417       1
5c3dcdb02adfc87d2161a95c4c42badd6c8b164b68d6f4f33003f3ae766812dc       1
19c67c8fce0ea76be83dd819f08aa90168dcceb69bec34c1a64c44d8edc22d7f       1
42d0211c3cbcba4fb7df2dcce583b7def774266d4cb130a8a6abe33729d44e00       1
d061fd0c3b824d39967202851ab4f5f27a3560785bd38e2b30cb050ada282a4b       1
Name: count, Length: 12868, dtype: int64


In [50]:
# Remove all rows with null values, and add to dropped rows dict
print(f'row count before dropping: {sth_yk.shape[0]}')

dropped_rows['No location'] = sth_yk.shape[0] - sth_yk.dropna().shape[0]

sth_yk = sth_yk.dropna()

print(f'row count after dropping: {sth_yk.shape[0]}')

row count before dropping: 14655
row count after dropping: 13987


In [51]:
for reason in dropped_rows:
    print(f'{reason}: {dropped_rows[reason]}')

duplicates: 658
No location: 668


***
Data is now clean.  
New crime ID will be made in another file.  
Function to clean will be as follows:

In [52]:
def Clean_Crime(raw_crime, dropped_rows):
    # Drop useless columns
    crime = raw_crime[['Crime ID', 'LSOA code', 'LSOA name', 'Month', 'Latitude', 'Longitude', 'Crime type']]

    # Rename columns
    crime = crime.rename(columns={
        'Crime ID': 'old_crime_id',
        'Month': 'date',
        'Latitude': 'latitude',
        'Longitude': 'longitude',
        'LSOA code': 'lsoa_code',
        'LSOA name': 'lsoa_name',
        'Crime type': 'crime_cat'
    })

    # Change month to date
    crime['date'] = pd.to_datetime(crime['date'], format='%Y-%m')

    # Drop duplicate rows
    dropped_rows['duplicates'] = crime.duplicated().sum()
    crime = crime.drop_duplicates()

    # Fill old crime ID with 'No ID' for crimes with no ID
    crime['old_crime_id'] = crime['old_crime_id'].fillna('No ID')

    # Remove rows with null locations
    dropped_rows['No location'] = crime.shape[0] - crime.dropna().shape[0]
    crime = crime.dropna()

    return crime
    
    